In [ ]:
from google.colab import files
files.upload()

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [ ]:
df = pd.read_csv('p4_wine.csv')
cols = list(df.drop(columns='target').columns)
X = StandardScaler().fit_transform(df[cols])
y = df.target.values

def knn(Xtr, ytr, Xq, k):
    d = np.linalg.norm(Xq[:, None] - Xtr[None], axis=2)
    idx = np.argsort(d, axis=1)[:, :k]
    return np.array([np.bincount(r).argmax() for r in ytr[idx]])

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
ks = range(1, 10)
acc = [(knn(Xtr, ytr, Xte, k) == yte).mean() for k in ks]
best = ks[int(np.argmax(acc))]
print('best k =', best, 'accuracy = %.3f' % max(acc))
plt.plot(ks, acc, marker='o')
plt.xlabel('k')
plt.ylabel('accuracy')
plt.show()

In [ ]:
c = [cols.index('alcohol'), cols.index('flavanoids')]
X2 = X[:, c]
gx, gy = np.meshgrid(np.linspace(X2[:, 0].min() - .5, X2[:, 0].max() + .5, 120), np.linspace(X2[:, 1].min() - .5, X2[:, 1].max() + .5, 120))
Z = knn(X2, y, np.c_[gx.ravel(), gy.ravel()], best).reshape(gx.shape)
plt.contourf(gx, gy, Z, alpha=.3)
plt.scatter(X2[:, 0], X2[:, 1], c=y, edgecolor='k')
plt.xlabel('alcohol')
plt.ylabel('flavanoids')
plt.title('kNN decision regions, k=%d' % best)
plt.show()

In [ ]:
rng = np.random.default_rng(0)
dims = [0, 5, 10, 25, 50, 100, 200]
res = []
for d in dims:
    Xn = np.c_[X, rng.normal(size=(len(X), d))]
    a, b, c1, c2 = train_test_split(Xn, y, test_size=0.3, random_state=42, stratify=y)
    res.append((knn(a, c1, b, 5) == c2).mean())
D = []
dd = [2, 10, 50, 100, 500, 1000]
for d in dd:
    P = rng.random((200, d))
    dist = np.linalg.norm(P[:1] - P[1:], axis=1)
    D.append((dist.max() - dist.min()) / dist.min())
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(dims, res, marker='o')
ax[0].set_xlabel('extra noise features')
ax[0].set_ylabel('accuracy')
ax[1].plot(dd, D, marker='o')
ax[1].set_xlabel('dimensions')
ax[1].set_ylabel('(max-min)/min distance')
plt.show()